# Persona Engine ML Lab

**Environment:** Google Colab / Local
**Purpose:** Preprocess raw conversation exports into clean, validated, split datasets with ChatML training examples.
**Input:** Raw conversation JSONL
**Output:** Cleaned JSONL, Train/Val/Test partitions, Tokenization-ready ChatML JSONL

In [ ]:
import sys
from pathlib import Path

ml_root = Path.cwd().resolve()
if ml_root.name != "ml" and (ml_root / "ml").is_dir():
    ml_root = ml_root / "ml"
if str(ml_root) not in sys.path:
    sys.path.insert(0, str(ml_root))

from src.config import load_config
from src.data import DatasetValidator, DataCleaner, DatasetSplitter, load_jsonl, save_jsonl
from src.preprocessing import ConversationBuilder

cfg = load_config()
input_data_path = Path(cfg.paths.sample_data) / "sample_conversations.jsonl"
print("Input Dataset:", input_data_path)

## 1. Validate Dataset Integrity

In [ ]:
validator = DatasetValidator()
result = validator.validate_file(input_data_path)
print(result.summary())
assert result.is_valid, "Dataset has validation errors!"

## 2. Personality-Preserving Cleaning

In [ ]:
raw_conversations = load_jsonl(input_data_path)
cleaner = DataCleaner()
cleaned_conversations = cleaner.clean_dataset(raw_conversations)
print(f"Original conversations: {len(raw_conversations)}")
print(f"Cleaned conversations : {len(cleaned_conversations)}")

## 3. Split Dataset into Train, Validation, and Test Sets

In [ ]:
splitter = DatasetSplitter(train_ratio=0.80, val_ratio=0.10, test_ratio=0.10, seed=42)
splits = splitter.split(cleaned_conversations)
print(splits.summary())

## 4. Build ChatML Training Examples

In [ ]:
builder = ConversationBuilder(chat_template="chatml")
train_examples = builder.build_dataset(splits.train, split="train")
print(f"Generated {len(train_examples)} training pairs.")

if train_examples:
    print("\nSample Training Example:")
    print("Conversation ID:", train_examples[0]["conversation_id"])
    print("Formatted Text:\n" + train_examples[0]["text"][:300] + "...\n")